# Final-token probes across training stages

At checkpoints $t\in\{512, 16384, 262144, 2097152\}$, train a separate linear probe at each residual-stream depth on **mixed binary/ternary trees**, and evaluate it only on an independent set of **fully ternary trees**. The target is the held-out final token and the representation is the last input position. The same probe datasets are reused at every checkpoint.


This notebook uses **12-layer transformers** (embedding width 256, 16 heads). Probe and figure filenames include `depth12` to keep these runs separate. Run all cells from the repository root.


## 1. Setup and probe datasets

In [ ]:
import gc, sys, pickle, warnings
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

WORKSPACE = Path.cwd()
if not (WORKSPACE / 'models').exists(): raise RuntimeError('Run from the repository root.')
sys.path[:0] = [str(WORKSPACE), str(WORKSPACE / 'datasets' / 'rhm')]
import models
from mixed_random_hierarchy_model import (
    sample_padded_rules, create_probabilities,
    sample_data_from_labels_varying_tree_tensorized,
)

V, L, BIN = 8, 3, 2
M_2, M_3, D_MAX, PAD_TOKEN = BIN, BIN * V, 3 ** L, V
T = D_MAX - 1
MODEL_DEPTH = 12
STEPS = [512, 16384, 262144, 2097152]
SEED_RULES, TRAIN_SEED, TEST_SEED = 12345678, 42, 43
N_TRAIN, N_TEST = 32768, 32768
NUM_HEADS, BATCH_SIZE, PROBE_C = 16, 256, 0.1
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
CHECKPOINTS = {step: WORKSPACE / f'run_L3_v8_bin2_ter16_P16777216_lr0.001_l20.03_bs128_depth12_demb_256_nheads_16_arr0_t{step}.pt' for step in STEPS}
RESULT_PATHS = {step: WORKSPACE / f'probe_results_step_{step}_mixed_train_ternary_test_last_token_depth12.pkl' for step in STEPS}
RECOMPUTE = False  # Reuse only the depth-12 result files.
# Cached stages need no checkpoint file.
missing = [CHECKPOINTS[step] for step in STEPS
           if (RECOMPUTE or not RESULT_PATHS[step].is_file())
           and not CHECKPOINTS[step].is_file()]
if missing: raise FileNotFoundError('Missing checkpoints:\n' + '\n'.join(map(str, missing)))

rules = sample_padded_rules(V, V, M_2, M_3, 2, 3, L, SEED_RULES)
mixed_probs = create_probabilities(M_2, M_3, L)
ternary_probs = {l: torch.cat([torch.zeros(M_2), torch.full((M_3,), 1/M_3)]) for l in range(L)}

def generate_sentences(n, probabilities, seed):
    state = torch.random.get_rng_state(); torch.manual_seed(seed)
    labels = torch.randint(0, V, (n,))
    sentences, _ = sample_data_from_labels_varying_tree_tensorized(labels, rules, probabilities, V, D_MAX)
    torch.random.set_rng_state(state)
    return sentences

def make_last_token_task(sentences):
    # Mirrors init.MySentenceDataLoader exactly.
    n = len(sentences); lengths = (sentences != PAD_TOKEN).sum(1)
    targets = sentences[torch.arange(n), lengths - 1]
    input_lengths = lengths - 1
    columns = torch.arange(T).unsqueeze(0).expand(n, T)
    source_columns = columns - T + input_lengths.unsqueeze(1)
    gathered = torch.gather(sentences[:, :T], 1, source_columns.clamp(min=0))
    inputs = torch.where(source_columns >= 0, gathered, torch.full_like(gathered, PAD_TOKEN))
    return inputs, targets, lengths

mixed_sentences = generate_sentences(N_TRAIN, mixed_probs, TRAIN_SEED)
ternary_sentences = generate_sentences(N_TEST, ternary_probs, TEST_SEED)
X_train_tokens, y_train, mixed_lengths = make_last_token_task(mixed_sentences)
X_test_tokens, y_test, ternary_lengths = make_last_token_task(ternary_sentences)
assert torch.all(ternary_lengths == D_MAX)
assert torch.any(mixed_lengths < D_MAX) and torch.any(mixed_lengths == D_MAX)
assert torch.all(y_train < V) and torch.all(y_test < V)
print(f'device={DEVICE}; mixed train={N_TRAIN:,}; ternary test={N_TEST:,}')
print(f'mixed lengths={mixed_lengths.min().item()}..{mixed_lengths.max().item()}; ternary lengths={ternary_lengths.unique().tolist()}')

## 2. Model loading, last-position activation extraction, and probing

In [ ]:
def infer_arch(state_dict):
    width = state_dict['token_embedding_table.weight'].shape[1]
    n_layers = max(int(k.split('.')[1]) for k in state_dict if k.startswith('blocks.')) + 1
    return dict(vocab_size=state_dict['token_embedding_table.weight'].shape[0],
                block_size=state_dict['blocks.0.attn.tril'].shape[0],
                embedding_dim=width, num_heads=NUM_HEADS,
                ffwd_size=state_dict['blocks.0.ffwd.net.0.weight'].shape[0] // width,
                num_layers=n_layers, dropout=0.0,
                rope='position_embedding_table.weight' not in state_dict, share_emb=False)

def load_model(path):
    saved = torch.load(path, map_location='cpu', weights_only=False)
    # Compiled checkpoints prefix parameter names with _orig_mod.
    state_dict = {key.removeprefix('_orig_mod.'): value
                  for key, value in saved['model'].items()}
    arch = infer_arch(state_dict); assert arch['block_size'] == T
    if (arch['num_layers'], arch['embedding_dim']) != (MODEL_DEPTH, 256):
        raise ValueError(f'Expected 12 layers and embedding width 256, got {arch}')
    model = models.CLM(**arch); model.load_state_dict(state_dict)
    return model.to(DEVICE).eval(), arch, int(saved.get('step', path.stem.rsplit('_t', 1)[-1]))

@torch.no_grad()
def extract_last_position_activations(model, inputs):
    # Shape: [embedding + blocks, examples, hidden width]. No other token positions are retained.
    by_depth = [[] for _ in range(model.num_layers + 1)]
    for start in range(0, len(inputs), BATCH_SIZE):
        batch = inputs[start:start + BATCH_SIZE].to(DEVICE); captured = {}
        hooks = [model.blocks[0].register_forward_pre_hook(
            lambda module, args: captured.update({0: args[0][:, -1].detach().cpu()}))]
        for depth, block in enumerate(model.blocks, 1):
            hooks.append(block.register_forward_hook(
                lambda module, args, output, depth=depth: captured.update({depth: output[:, -1].detach().cpu()})))
        model(batch)
        for hook in hooks: hook.remove()
        for depth in range(model.num_layers + 1): by_depth[depth].append(captured[depth].numpy())
    return np.stack([np.concatenate(chunks) for chunks in by_depth])

def probe_across_depth(train_acts, test_acts):
    results = {'train_accuracy': [], 'test_accuracy': [],
               'train_cross_entropy_normalized': [], 'test_cross_entropy_normalized': []}
    train_labels, test_labels = y_train.numpy(), y_test.numpy()
    for depth, (X_train, X_test) in enumerate(zip(train_acts, test_acts)):
        probe = Pipeline([('scaler', StandardScaler()), ('classifier', LogisticRegression(
            C=PROBE_C, solver='lbfgs', max_iter=5000, random_state=0))])
        probe.fit(X_train, train_labels)
        results['train_accuracy'].append(probe.score(X_train, train_labels))
        results['test_accuracy'].append(probe.score(X_test, test_labels))
        train_ce = log_loss(train_labels, probe.predict_proba(X_train), labels=np.arange(V)) / np.log(V)
        test_ce = log_loss(test_labels, probe.predict_proba(X_test), labels=np.arange(V)) / np.log(V)
        results['train_cross_entropy_normalized'].append(train_ce)
        results['test_cross_entropy_normalized'].append(test_ce)
        print(f"  depth {depth}: acc train={results['train_accuracy'][-1]:.4f}, test={results['test_accuracy'][-1]:.4f}; "
              f"CE/log(V) train={train_ce:.4f}, test={test_ce:.4f}")
    counts = np.bincount(train_labels, minlength=V)
    results.update(chance_uniform=1/V, train_majority_baseline=counts.max()/counts.sum())
    return results

## 3. Run missing checkpoints only

With `RECOMPUTE = False`, reuse the saved results for 512, 16384, and 262144 and fit only the new final checkpoint, 2097152. The old final-checkpoint results are preserved and never relabeled.


In [ ]:
token_probe_results = {}
architecture = None
for requested_step in STEPS:
    print(f'\n=== checkpoint t={requested_step} ===')
    if RESULT_PATHS[requested_step].exists() and not RECOMPUTE:
        with RESULT_PATHS[requested_step].open('rb') as handle: cached = pickle.load(handle)
        cached_result = cached.get('token_probe', cached)
        if ('test_cross_entropy_normalized' in cached_result
                and all(len(cached_result.get(metric, [])) == MODEL_DEPTH + 1
                        for metric in ('test_accuracy', 'test_cross_entropy_normalized'))):
            token_probe_results[requested_step] = np.asarray(cached_result['test_accuracy'], dtype=float)
            print('reusing:', RESULT_PATHS[requested_step])
            continue
        print('cached result has no cross-entropy; refitting this checkpoint')
    model, arch, loaded_step = load_model(CHECKPOINTS[requested_step])
    if loaded_step != requested_step: raise ValueError(f'Expected step {requested_step}, checkpoint reports {loaded_step}')
    if architecture is None: architecture = arch
    elif (arch['num_layers'], arch['embedding_dim']) != (architecture['num_layers'], architecture['embedding_dim']):
        raise ValueError('Checkpoint architectures differ.')
    train_acts = extract_last_position_activations(model, X_train_tokens)
    test_acts = extract_last_position_activations(model, X_test_tokens)
    result = probe_across_depth(train_acts, test_acts)
    token_probe_results[requested_step] = np.asarray(result['test_accuracy'], dtype=float)
    payload = {'token_probe': result, 'checkpoint_step': requested_step, 'probe_C': PROBE_C,
               'n_train_mixed': N_TRAIN, 'n_test_ternary': N_TEST,
               'train_seed': TRAIN_SEED, 'test_seed': TEST_SEED,
               'probe_position': 'last input position'}
    with RESULT_PATHS[requested_step].open('wb') as handle: pickle.dump(payload, handle)
    print('saved:', RESULT_PATHS[requested_step])
    del model, train_acts, test_acts; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    elif DEVICE == 'mps': torch.mps.empty_cache()

## 4. Paper figure: final-token probes and theoretical plateaus

Solid curves show mixed-train/ternary-test final-token results. Horizontal segments show theoretical posterior accuracies. The Inside/Outside divider is at L6.


In [ ]:
plt.rcParams.update({'font.size': 9, 'axes.labelsize': 10, 'axes.titlesize': 10,
                     'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
                     'legend.fontsize': 8, 'legend.title_fontsize': 8.5,
                     'lines.linewidth': 1.8, 'pdf.fonttype': 42, 'ps.fonttype': 42})
stage_labels = {512: 'Stage 1', 16384: 'Stage 2', 262144: 'Stage 3', 2097152: 'Stage 4'}
theoretical_accuracy = {512: 0.417308, 16384: 0.507880, 262144: 0.737, 2097152: 0.851}
cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
colors = {step: cycle[i] for i, step in enumerate(STEPS)}

# Reload saved results, making this plotting cell independently rerunnable.
token_probe_results = {}
for step in STEPS:
    with RESULT_PATHS[step].open('rb') as handle: saved = pickle.load(handle)
    saved_result = saved.get('token_probe', saved)
    token_probe_results[step] = np.asarray(saved_result['test_accuracy'], dtype=float)

all_curves = list(token_probe_results.values())
n_depths = {len(curve) for curve in all_curves}
if n_depths != {MODEL_DEPTH + 1}: raise ValueError(f'Expected 13 probe depths (emb, L1–L12), got {n_depths}')
depth = np.arange(n_depths.pop()); layer_labels = ['emb'] + [f'L{k}' for k in range(1, len(depth))]
L6_INDEX = layer_labels.index('L6')
fig, ax = plt.subplots(figsize=(7.0, 3.8))
for step in STEPS:
    ax.plot(depth, token_probe_results[step], color=colors[step], ls='-', marker='o', lw=1.9, ms=4.5, alpha=.95, zorder=4)

x_right = depth[-1]
for step in STEPS:
    # Retain the requested left-shifted Stage-1 plateau.
    x_start, x_end = ((x_right - 4.4, x_right - 4.1) if step == 512 else (x_right + .10, x_right + .40))
    ax.hlines(theoretical_accuracy[step], x_start, x_end, color=colors[step], lw=2.2, zorder=5)
ax.axhline(1/V, color='grey', ls=':', lw=1.2, zorder=1)
ax.axvline(L6_INDEX, color='black', ls='--', lw=1, alpha=.55, zorder=2)
ax.text(L6_INDEX-.18, .97, 'Inside', ha='right', va='top', fontsize=9)
ax.text(L6_INDEX+.18, .97, 'Outside', ha='left', va='top', fontsize=9)
ax.set(xticks=depth, xticklabels=layer_labels, xlim=(depth[0]-.2, depth[-1]+.55), ylim=(0, 1.02),
       xlabel='Transformer depth', ylabel='Probe test accuracy')
ax.grid(alpha=.18, lw=.6)
stage_handles = [Line2D([0], [0], color=colors[s], lw=2, label=stage_labels[s]) for s in STEPS]
ax.legend(handles=stage_handles, title='Training stage', loc='lower right', frameon=False, handlelength=2.4)
fig.tight_layout()
pdf_path = WORKSPACE / 'token_and_level2_probe_mixed_train_ternary_test_across_depth_depth12.pdf'
png_path = WORKSPACE / 'token_and_level2_probe_mixed_train_ternary_test_across_depth_depth12.png'
fig.savefig(pdf_path, format='pdf', bbox_inches='tight'); fig.savefig(png_path, dpi=300, bbox_inches='tight')
print('saved:', pdf_path); print('saved:', png_path); plt.show()

## 5. Normalized cross-entropy across depth

This figure uses the ternary-test cross-entropy of each mixed-data linear probe, divided by $\log V$. Thus a uniform predictive distribution has normalized loss 1 and lower is better. The theoretical plateaus are associated in order with Stage 1 ($L_{\mathrm{trig}}$), Stage 2 ($L_1$), Stage 3 ($L_{2,\mathrm{VT}}$), and Stage 4 ($L_3$).


In [ ]:
theoretical_loss_normalized = {
    512: 0.628652921,          # L_trig
    16384: 0.452838292701,  # L_1
    262144: 0.20574168745407478,  # L_2_VT
    2097152: 0.1159764843808,     # L_3
}
test_ce_results = {}
for step in STEPS:
    with RESULT_PATHS[step].open('rb') as handle: saved = pickle.load(handle)
    saved_result = saved.get('token_probe', saved)
    if 'test_cross_entropy_normalized' not in saved_result:
        raise KeyError(f'{RESULT_PATHS[step].name} predates loss measurement; rerun Section 3.')
    test_ce_results[step] = np.asarray(saved_result['test_cross_entropy_normalized'], dtype=float)

n_depths = {len(curve) for curve in test_ce_results.values()}
if n_depths != {MODEL_DEPTH + 1}: raise ValueError(f'Expected 13 probe depths (emb, L1–L12), got {n_depths}')
depth = np.arange(n_depths.pop())
layer_labels = ['emb'] + [f'L{k}' for k in range(1, len(depth))]
L6_INDEX = layer_labels.index('L6')
fig, ax = plt.subplots(figsize=(7.0, 3.8))
for step in STEPS:
    ax.plot(depth, test_ce_results[step], color=colors[step], marker='o', lw=1.9, ms=4.5,
            alpha=.95, zorder=4, label=stage_labels[step])
x_right = depth[-1]
for step in STEPS:
    x_start, x_end = ((x_right - 4.4, x_right - 4.1) if step == 512 else (x_right + .10, x_right + .40))
    ax.hlines(theoretical_loss_normalized[step], x_start, x_end, color=colors[step], lw=2.2, zorder=5)
ax.axhline(1.0, color='grey', ls=':', lw=1.2, zorder=1, label='Uniform prediction')
ax.axvline(L6_INDEX, color='black', ls='--', lw=1, alpha=.55, zorder=2)
y_top = max(1.02, max(curve.max() for curve in test_ce_results.values()) * 1.05)
ax.text(L6_INDEX-.18, y_top*.97, 'Inside', ha='right', va='top', fontsize=9)
ax.text(L6_INDEX+.18, y_top*.97, 'Outside', ha='left', va='top', fontsize=9)
ax.set(xticks=depth, xticklabels=layer_labels, xlim=(depth[0]-.2, depth[-1]+.55), ylim=(0, y_top),
       xlabel='Transformer depth', ylabel=r'Ternary-test cross-entropy / $\log V$')
ax.grid(alpha=.18, lw=.6)
stage_handles = [Line2D([0], [0], color=colors[s], marker='o', lw=2, label=stage_labels[s]) for s in STEPS]
ax.legend(handles=stage_handles, title='Training stage', loc='upper right', frameon=False, handlelength=2.4)
fig.tight_layout()
loss_pdf_path = WORKSPACE / 'token_probe_mixed_train_ternary_test_cross_entropy_across_depth_depth12.pdf'
loss_png_path = WORKSPACE / 'token_probe_mixed_train_ternary_test_cross_entropy_across_depth_depth12.png'
fig.savefig(loss_pdf_path, format='pdf', bbox_inches='tight')
fig.savefig(loss_png_path, dpi=300, bbox_inches='tight')
print('saved:', loss_pdf_path); print('saved:', loss_png_path); plt.show()

## 6. Log-scale cross-entropy with annotated theoretical plateaus

This alternative presentation keeps the empirical stage colors and displays the four theoretical normalized losses as black dashed, annotated horizontal lines.

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 3.8))
for step in STEPS:
    ax.plot(
        depth, test_ce_results[step], color=colors[step], marker='o',
        lw=1.9, ms=4.5, alpha=.95, zorder=4, label=stage_labels[step],
    )

L_trig = theoretical_loss_normalized[512]
L_1 = theoretical_loss_normalized[16384]
L_2_VT = theoretical_loss_normalized[262144]
L_3 = theoretical_loss_normalized[2097152]
plateaus = [
    (L_trig, 'black'),
    (L_1, 'black'),
    (L_2_VT, 'black'),
    (L_3, 'black'),
]
for loss, color in plateaus:
    ax.axhline(loss, color=color, ls='--', lw=1.35, alpha=.8, zorder=2)

# Multiplicative offsets are appropriate on a logarithmic axis.
x_text = depth[-1] + 0.18
offset = 0.92
ax.text(x_text, L_trig * offset * 1.10, r'$\mathcal{L}_{\mathrm{trig}}^{\mathrm{VT}}$',
        color='black', fontsize=10, va='top')
ax.text(x_text, L_1 * offset, r'$\mathcal{L}_{\mathrm{trig}}^{\mathrm{FT}}$',
        color='black', fontsize=10, va='top')
ax.text(x_text, L_2_VT * offset * 1.45, r'$\mathcal{L}_{2}^{\mathrm{VT}}$',
        color='black', fontsize=10, va='top')
ax.text(x_text, L_3 * offset * 1.05, r'$\mathcal{L}_{3}^{\mathrm{FT}}$',
        color='black', fontsize=10, va='top')

ax.axhline(1.0, color='grey', ls=':', lw=1.2, zorder=1)
ax.axvline(L6_INDEX, color='black', ls='--', lw=1, alpha=.55, zorder=2)
ax.set_yscale('log')
positive_values = np.concatenate([*test_ce_results.values(), np.asarray(list(theoretical_loss_normalized.values()))])
y_bottom = positive_values.min() * 0.72
y_top = max(1.08, positive_values.max() * 1.12)
ax.set(xticks=depth, xticklabels=layer_labels,
       xlim=(depth[0] - .2, depth[-1] + 1.18), ylim=(y_bottom, y_top),
       xlabel='Transformer depth', ylabel=r'Ternary-test cross-entropy / $\log V$')
ax.text(L6_INDEX - .18, y_top / 1.05, 'Inside', ha='right', va='top', fontsize=9)
ax.text(L6_INDEX + .18, y_top / 1.05, 'Outside', ha='left', va='top', fontsize=9)
ax.grid(which='both', alpha=.18, lw=.6)
stage_handles = [
    Line2D([0], [0], color=colors[s], marker='o', lw=2, label=stage_labels[s])
    for s in STEPS
]
ax.legend(handles=stage_handles, title='Training stage', loc='upper right',
          frameon=False, handlelength=2.4)
fig.tight_layout()
log_loss_pdf_path = WORKSPACE / 'token_probe_mixed_train_ternary_test_cross_entropy_across_depth_log_depth12.pdf'
log_loss_png_path = WORKSPACE / 'token_probe_mixed_train_ternary_test_cross_entropy_across_depth_log_depth12.png'
fig.savefig(log_loss_pdf_path, format='pdf', bbox_inches='tight')
fig.savefig(log_loss_png_path, dpi=300, bbox_inches='tight')
print('saved:', log_loss_pdf_path)
print('saved:', log_loss_png_path)
plt.show()

## 7. Regenerate the final figure from saved data
Run the following cell on its own, including after a kernel restart. It requires NumPy, Matplotlib, and the four saved mixed-train/ternary-test token-probe files in `WORKSPACE`. It exports the combined accuracy and cross-entropy figure.


In [ ]:
# Run this cell directly after restarting the kernel; no model/probe training needed.
from pathlib import Path
import pickle
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

WORKSPACE = Path.cwd()  # Set to the repository directory if running elsewhere.
V = 8
MODEL_DEPTH = 12
STEPS = [512, 16384, 262144, 2097152]
RESULT_PATHS = {
    step: WORKSPACE / f'probe_results_step_{step}_mixed_train_ternary_test_last_token_depth12.pkl'
    for step in STEPS
}
required_paths = list(RESULT_PATHS.values())
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing saved probe data; check WORKSPACE:\n' + '\n'.join(missing))

plt.rcParams.update({'font.size': 9, 'axes.labelsize': 10, 'axes.titlesize': 10,
                     'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
                     'legend.fontsize': 8, 'legend.title_fontsize': 8.5,
                     'lines.linewidth': 1.8, 'pdf.fonttype': 42, 'ps.fonttype': 42})
stage_labels = {512: 'Stage 1', 16384: 'Stage 2', 262144: 'Stage 3', 2097152: 'Stage 4'}
theoretical_accuracy = {512: 0.417308, 16384: 0.507880, 262144: 0.737, 2097152: 0.851}
cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
colors = {step: cycle[i] for i, step in enumerate(STEPS)}

# Reload saved results, making this plotting cell independently rerunnable.
token_probe_results = {}
for step in STEPS:
    with RESULT_PATHS[step].open('rb') as handle: saved = pickle.load(handle)
    saved_result = saved.get('token_probe', saved)
    token_probe_results[step] = np.asarray(saved_result['test_accuracy'], dtype=float)

all_curves = list(token_probe_results.values())
n_depths = {len(curve) for curve in all_curves}
if n_depths != {MODEL_DEPTH + 1}: raise ValueError(f'Expected 13 probe depths (emb, L1–L12), got {n_depths}')
depth = np.arange(n_depths.pop()); layer_labels = ['emb'] + [f'L{k}' for k in range(1, len(depth))]
L6_INDEX = layer_labels.index('L6')

theoretical_loss_normalized = {
    512: 0.628652921,          # L_trig
    16384: 0.452838292701,  # L_1
    262144: 0.20574168745407478,  # L_2_VT
    2097152: 0.1159764843808,     # L_3
}
test_ce_results = {}
for step in STEPS:
    with RESULT_PATHS[step].open('rb') as handle: saved = pickle.load(handle)
    saved_result = saved.get('token_probe', saved)
    if 'test_cross_entropy_normalized' not in saved_result:
        raise KeyError(f'{RESULT_PATHS[step].name} predates loss measurement; rerun Section 3.')
    test_ce_results[step] = np.asarray(saved_result['test_cross_entropy_normalized'], dtype=float)

# Validate both panels against the same depth axis before plotting.
for step in STEPS:
    for metric, curve in [('accuracy', token_probe_results[step]), ('cross-entropy', test_ce_results[step])]:
        if curve.shape != depth.shape or not np.all(np.isfinite(curve)):
            raise ValueError(f'Invalid {metric} curve at checkpoint {step}: expected {depth.shape}.')
    if np.any(test_ce_results[step] <= 0):
        raise ValueError(f'Cross-entropy must be positive for the log axis (checkpoint {step}).')

L_trig, L_1, L_2_VT, L_3 = (theoretical_loss_normalized[step] for step in STEPS)

# Two-panel, double-column paper figure: accuracy (left), normalized loss (right).
fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(7.15, 3.45), sharex=True)

for step in STEPS:
    ax_acc.plot(depth, token_probe_results[step], color=colors[step], ls='-', marker='o', lw=1.8, ms=3.8, alpha=.95, zorder=4)
    ax_loss.plot(depth, test_ce_results[step], color=colors[step], marker='o', lw=1.8, ms=3.8, alpha=.95, zorder=4)

accuracy_plateaus = [
    (theoretical_accuracy[512], r'$A_{\mathrm{trig}}^{\mathrm{VT}}$'),
    (theoretical_accuracy[16384], r'$A_{\mathrm{trig}}^{\mathrm{FT}}$'),
    (theoretical_accuracy[262144], r'$A_{2}^{\mathrm{VT}}$'),
    (theoretical_accuracy[2097152], r'$A_{3}^{\mathrm{FT}}$'),
]
loss_plateaus = [
    (L_trig, r'$\mathcal{L}_{\mathrm{trig}}^{\mathrm{VT}}$', 1.10),
    (L_1, r'$\mathcal{L}_{\mathrm{trig}}^{\mathrm{FT}}$', 1.00),
    (L_2_VT, r'$\mathcal{L}_{2}^{\mathrm{VT}}$', 1.45),
    (L_3, r'$\mathcal{L}_{3}^{\mathrm{FT}}$', 1.05),
]
x_text = depth[-1] + .13
for value, label in accuracy_plateaus:
    ax_acc.axhline(value, color='black', ls='--', lw=1.05, alpha=.72, zorder=2)
    ax_acc.text(x_text, value + .008, label, color='black', fontsize=8.5, va='bottom')
for value, label, multiplier in loss_plateaus:
    ax_loss.axhline(value, color='black', ls='--', lw=1.05, alpha=.72, zorder=2)
    ax_loss.text(x_text, value * .92 * multiplier, label, color='black', fontsize=8.5, va='top')

ax_acc.axhline(1/V, color='grey', ls=':', lw=1.0, zorder=1)
ax_loss.axhline(1.0, color='grey', ls=':', lw=1.0, zorder=1)
ax_loss.set_yscale('log')
loss_values = np.concatenate([*test_ce_results.values(), np.asarray(list(theoretical_loss_normalized.values()))])
ax_acc.set_ylim(0, 1.02)
ax_loss.set_ylim(loss_values.min() * .72, max(1.08, loss_values.max() * 1.12))

for panel in (ax_acc, ax_loss):
    panel.axvline(L6_INDEX, color='black', ls='--', lw=.9, alpha=.5, zorder=2)
    panel.set_xticks(depth, layer_labels)
    panel.set_xlim(depth[0] - .2, depth[-1] + 1.55)
    panel.set_xlabel('Transformer depth')
    panel.grid(which='both', alpha=.16, lw=.55)
    panel.text(L6_INDEX - .16, .975, 'Inside', transform=panel.get_xaxis_transform(), ha='right', va='top', fontsize=8)
    panel.text(L6_INDEX + .16, .975, 'Outside', transform=panel.get_xaxis_transform(), ha='left', va='top', fontsize=8)

ax_acc.set_title('(a) Probe accuracy')
ax_loss.set_title('(b) Probe cross-entropy')
ax_acc.set_ylabel('Ternary-test accuracy')
ax_loss.set_ylabel(r'Ternary-test cross-entropy / $\log V$')
stage_handles = [Line2D([0], [0], color=colors[s], marker='o', lw=1.8, ms=3.8, label=stage_labels[s]) for s in STEPS]
fig.legend(handles=stage_handles, loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(.5, 1.01), handlelength=2.0, columnspacing=1.25)
fig.tight_layout(rect=(0, 0, 1, .91), w_pad=1.35)
combined_pdf_path = WORKSPACE / 'token_probe_accuracy_and_cross_entropy_across_depth_depth12.pdf'
combined_png_path = WORKSPACE / 'token_probe_accuracy_and_cross_entropy_across_depth_depth12.png'
fig.savefig(combined_pdf_path, format='pdf', bbox_inches='tight')
fig.savefig(combined_png_path, dpi=300, bbox_inches='tight')
print('saved:', combined_pdf_path)
print('saved:', combined_png_path)
plt.show()

## 8. Accuracy panel only
Run this cell independently from a fresh kernel. It loads the saved accuracy data and exports a 3.5 × 3.45 inch panel with a two-row stage legend, suitable for pairing with another panel across two columns. Adjust `FIGSIZE` to match the companion figure.

In [ ]:
# Run this cell directly after restarting the kernel; no model/probe training needed.
from pathlib import Path
import pickle
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

WORKSPACE = Path.cwd()  # Set to the repository directory if running elsewhere.
V = 8
MODEL_DEPTH = 12
STEPS = [512, 16384, 262144, 2097152]
RESULT_PATHS = {
    step: WORKSPACE / f'probe_results_step_{step}_mixed_train_ternary_test_last_token_depth12.pkl'
    for step in STEPS
}
required_paths = list(RESULT_PATHS.values())
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing saved probe data; check WORKSPACE:\n' + '\n'.join(missing))

plt.rcParams.update({'font.size': 9, 'axes.labelsize': 10, 'axes.titlesize': 10,
                     'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
                     'legend.fontsize': 8, 'legend.title_fontsize': 8.5,
                     'lines.linewidth': 1.8, 'pdf.fonttype': 42, 'ps.fonttype': 42})
stage_labels = {512: 'Stage 1', 16384: 'Stage 2', 262144: 'Stage 3', 2097152: 'Stage 4'}
theoretical_accuracy = {512: 0.417308, 16384: 0.507880, 262144: 0.737, 2097152: 0.851}
cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
colors = {step: cycle[i] for i, step in enumerate(STEPS)}

# Reload saved results, making this plotting cell independently rerunnable.
token_probe_results = {}
for step in STEPS:
    with RESULT_PATHS[step].open('rb') as handle: saved = pickle.load(handle)
    saved_result = saved.get('token_probe', saved)
    token_probe_results[step] = np.asarray(saved_result['test_accuracy'], dtype=float)

all_curves = list(token_probe_results.values())
n_depths = {len(curve) for curve in all_curves}
if n_depths != {MODEL_DEPTH + 1}: raise ValueError(f'Expected 13 probe depths (emb, L1–L12), got {n_depths}')
depth = np.arange(n_depths.pop()); layer_labels = ['emb'] + [f'L{k}' for k in range(1, len(depth))]
L6_INDEX = layer_labels.index('L6')

# Single-column panel for pairing within a double-column figure.
FIGSIZE = (3.5, 3.45)  # inches; adjust to match the companion figure.
fig, ax_acc = plt.subplots(figsize=FIGSIZE)
for step in STEPS:
    ax_acc.plot(depth, token_probe_results[step], color=colors[step], ls='-', marker='o', lw=1.8, ms=3.8, alpha=.95, zorder=4)

accuracy_plateaus = [
    (theoretical_accuracy[512], r'$A_{\mathrm{trig}}^{\mathrm{VT}}$'),
    (theoretical_accuracy[16384], r'$A_{\mathrm{trig}}^{\mathrm{FT}}$'),
    (theoretical_accuracy[262144], r'$A_{2}^{\mathrm{VT}}$'),
    (theoretical_accuracy[2097152], r'$A_{3}^{\mathrm{FT}}$'),
]
x_text = depth[-1] + .13
for value, label in accuracy_plateaus:
    ax_acc.axhline(value, color='black', ls='--', lw=1.05, alpha=.72, zorder=2)
    ax_acc.text(x_text, value + .008, label, color='black', fontsize=8.5, va='bottom')
ax_acc.axhline(1/V, color='grey', ls=':', lw=1.0, zorder=1)
ax_acc.axvline(L6_INDEX, color='black', ls='--', lw=.9, alpha=.5, zorder=2)
ax_acc.set(xticks=depth, xticklabels=layer_labels,
           xlim=(depth[0] - .2, depth[-1] + 1.55), ylim=(0, 1.02),
           xlabel='Transformer depth', ylabel='Ternary-test accuracy')
ax_acc.set_title(r'Probing for depth $D=12$', pad=8)
ax_acc.grid(which='both', alpha=.16, lw=.55)
ax_acc.text(L6_INDEX - .16, .975, 'Inside', transform=ax_acc.get_xaxis_transform(), ha='right', va='top', fontsize=8)
ax_acc.text(L6_INDEX + .16, .975, 'Outside', transform=ax_acc.get_xaxis_transform(), ha='left', va='top', fontsize=8)
# Matplotlib fills columns first: this order displays stages 1/2 above 3/4.
legend_steps = [STEPS[i] for i in (0, 2, 1, 3)]
stage_handles = [Line2D([0], [0], color=colors[s], marker='o', lw=1.8, ms=3.8, label=stage_labels[s]) for s in legend_steps]
fig.legend(handles=stage_handles, loc='upper center', ncol=2, frameon=False,
           bbox_to_anchor=(.5, .995), handlelength=2.0, columnspacing=1.6)
fig.tight_layout(rect=(0, 0, 1, .86))
# Preserve the exact figure dimensions for predictable placement in the paper.
pdf_path = WORKSPACE / 'token_probe_accuracy_across_depth_depth12.pdf'
png_path = WORKSPACE / 'token_probe_accuracy_across_depth_depth12.png'
fig.savefig(pdf_path, format='pdf')
fig.savefig(png_path, dpi=300)
print('saved:', pdf_path)
print('saved:', png_path)
plt.show()


In [ ]:
# =====================================================================
# Standalone cross-entropy panel with legend outside
# =====================================================================

from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

WORKSPACE = Path.cwd()

V = 8
MODEL_DEPTH = 12
STEPS = [512, 16384, 262144, 2097152]

RESULT_PATHS = {
    step: WORKSPACE / f'probe_results_step_{step}_mixed_train_ternary_test_last_token_depth12.pkl'
    for step in STEPS
}

# ---------------------------------------------------------------------
# Plot style
# ---------------------------------------------------------------------
plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'xtick.labelsize': 8.5,
    'ytick.labelsize': 8.5,
    'legend.fontsize': 8,
    'lines.linewidth': 1.8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

stage_labels = {
    512: 'Stage 1',
    16384: 'Stage 2',
    262144: 'Stage 3',
    2097152: 'Stage 4',
}

cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
colors = {step: cycle[i] for i, step in enumerate(STEPS)}

# ---------------------------------------------------------------------
# Theoretical plateaus
# ---------------------------------------------------------------------
theoretical_loss_normalized = {
    512: 0.628652921,               # L_trig
    16384: 0.452838292701,       # L_1
    262144: 0.20574168745407478, # L_2_VT
    2097152: 0.1159764843808,    # L_3
}

# ---------------------------------------------------------------------
# Load cross-entropy curves
# ---------------------------------------------------------------------
test_ce_results = {}

for step in STEPS:
    path = RESULT_PATHS[step]

    if not path.is_file():
        raise FileNotFoundError(f'Missing saved probe data: {path}')

    with path.open('rb') as handle:
        saved = pickle.load(handle)

    saved_result = saved.get('token_probe', saved)

    if 'test_cross_entropy_normalized' not in saved_result:
        raise KeyError(
            f'{path.name} predates loss measurement; '
            'rerun the probe section that measures cross-entropy.'
        )

    test_ce_results[step] = np.asarray(
        saved_result['test_cross_entropy_normalized'],
        dtype=float
    )

# ---------------------------------------------------------------------
# Depth axis
# ---------------------------------------------------------------------
n_depths = {len(curve) for curve in test_ce_results.values()}
if n_depths != {MODEL_DEPTH + 1}:
    raise ValueError(f'Expected 13 probe depths (emb, L1–L12), got {n_depths}')

depth = np.arange(n_depths.pop())
layer_labels = ['emb'] + [f'L{k}' for k in range(1, len(depth))]
L6_INDEX = layer_labels.index('L6')

for step, curve in test_ce_results.items():
    if curve.shape != depth.shape:
        raise ValueError(
            f'Invalid cross-entropy curve at checkpoint {step}: '
            f'expected shape {depth.shape}, got {curve.shape}.'
        )
    if not np.all(np.isfinite(curve)):
        raise ValueError(f'Non-finite cross-entropy values at checkpoint {step}.')
    if np.any(curve <= 0):
        raise ValueError(
            f'Cross-entropy must be positive for the log axis '
            f'(checkpoint {step}).'
        )

# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(3.35, 3.2))

for step in STEPS:
    ax.plot(
        depth,
        test_ce_results[step],
        color=colors[step],
        marker='o',
        lw=1.8,
        ms=3.8,
        alpha=.95,
        zorder=4,
    )

# ---------------------------------------------------------------------
# Theoretical loss plateaus
# ---------------------------------------------------------------------
L_trig = theoretical_loss_normalized[512]
L_1 = theoretical_loss_normalized[16384]
L_2_VT = theoretical_loss_normalized[262144]
L_3 = theoretical_loss_normalized[2097152]

loss_plateaus = [
    (L_trig,  r'$\mathcal{L}_{\mathrm{trig}}^{\mathrm{VT}}$', 1.10),
    (L_1,     r'$\mathcal{L}_{\mathrm{trig}}^{\mathrm{FT}}$', 1.00),
    (L_2_VT,  r'$\mathcal{L}_{2}^{\mathrm{VT}}$',             1.45),
    (L_3,     r'$\mathcal{L}_{3}^{\mathrm{FT}}$',             1.05),
]

x_text = depth[-1] + 0.13

for value, label, multiplier in loss_plateaus:
    ax.axhline(
        value,
        color='black',
        ls='--',
        lw=1.05,
        alpha=.72,
        zorder=2,
    )
    ax.text(
        x_text,
        value * .92 * multiplier,
        label,
        color='black',
        fontsize=8.5,
        va='top',
    )

# Reference level
ax.axhline(
    1.0,
    color='grey',
    ls=':',
    lw=1.0,
    zorder=1,
)

# ---------------------------------------------------------------------
# Inside / outside division
# ---------------------------------------------------------------------
ax.axvline(
    L6_INDEX,
    color='black',
    ls='--',
    lw=.9,
    alpha=.5,
    zorder=2,
)

ax.text(
    L6_INDEX - .16,
    .975,
    'Inside',
    transform=ax.get_xaxis_transform(),
    ha='right',
    va='top',
    fontsize=8,
)

ax.text(
    L6_INDEX + .16,
    .975,
    'Outside',
    transform=ax.get_xaxis_transform(),
    ha='left',
    va='top',
    fontsize=8,
)

# ---------------------------------------------------------------------
# Axes
# ---------------------------------------------------------------------
ax.set_title('Depth pruning')

ax.set_yscale('log')

loss_values = np.concatenate([
    *test_ce_results.values(),
    np.asarray(list(theoretical_loss_normalized.values()))
])

ax.set_ylim(
    loss_values.min() * .72,
    max(1.08, loss_values.max() * 1.12),
)

ax.set_xticks(depth, layer_labels)
ax.set_xlim(depth[0] - .2, depth[-1] + 1.55)

ax.set_xlabel('Transformer depth')
ax.set_ylabel(r'$\mathcal{L}_{\mathrm{pruned}} / \ln(v)$')

ax.grid(which='both', alpha=.16, lw=.55)
# ---------------------------------------------------------------------
# Legend outside the axes
# ---------------------------------------------------------------------
stage_handles = [
    Line2D(
        [0], [0],
        color=colors[step],
        marker='o',
        lw=1.8,
        ms=3.8,
        label=stage_labels[step],
    )
    for step in STEPS
]

fig.legend(
    handles=stage_handles,
    loc='upper center',
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.02),
    handlelength=2.0,
    columnspacing=1.2,
)

# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------
fig.tight_layout(rect=(0, 0, 1, 0.92))

pdf_path = WORKSPACE / 'depth_pruning_depth12.pdf'
png_path = WORKSPACE / 'depth_pruning_depth12.png'

fig.savefig(pdf_path, format='pdf', bbox_inches='tight')
fig.savefig(png_path, dpi=300, bbox_inches='tight')

print('saved:', pdf_path)
print('saved:', png_path)

plt.show()

## 9. Rerun only the corrected final checkpoint and refresh the plots

Run the next cell alone, including after restarting the kernel. It fits only **t=2097152**, reuses the three earlier saved stages, and regenerates the combined figure, accuracy panel, and cross-entropy panel. If the new final result already exists, it only refreshes the plots. Supply the matching **12-layer, width-256, 16-head** checkpoint at the path shown in any missing-file error.


In [ ]:
# Run this cell alone from a fresh kernel at the repository root.
# Reuse the first three saved stages; fit only the corrected final stage.
import json
from pathlib import Path

notebook_path = Path.cwd() / 'probing_inside_outside_ternary_mixed_train_test_token_level_depth_12.ipynb'
notebook = json.loads(notebook_path.read_text())
checkpoint_path = Path.cwd() / 'run_L3_v8_bin2_ter16_P16777216_lr0.001_l20.03_bs128_depth12_demb_256_nheads_16_arr0_t2097152.pt'
final_result_path = Path.cwd() / 'probe_results_step_2097152_mixed_train_ternary_test_last_token_depth12.pkl'
required_caches = [Path.cwd() / f'probe_results_step_{step}_mixed_train_ternary_test_last_token_depth12.pkl'
                   for step in (512, 16384, 262144)]
missing_caches = [str(path) for path in required_caches if not path.is_file()]
if missing_caches:
    raise FileNotFoundError('Earlier-stage results are required for a final-stage-only rerun:\n' + '\n'.join(missing_caches))
if not final_result_path.is_file() and not checkpoint_path.is_file():
    raise FileNotFoundError(f'Place the matching depth-12 checkpoint here before running:\n{checkpoint_path}')

# Use a separate namespace so this cell also works in an existing kernel.
run_namespace = {'__name__': '__main__'}
def run_notebook_cell(index):
    source = ''.join(notebook['cells'][index]['source'])
    exec(compile(source, f'{notebook_path.name}:cell-{index}', 'exec'), run_namespace)

if not final_result_path.is_file():
    run_notebook_cell(2)  # Recreate the identical seeded probe datasets.
    run_notebook_cell(4)  # Model loading and probe functions.
    run_namespace['STEPS'] = [2097152]  # Never fit an earlier checkpoint here.
    run_notebook_cell(6)  # Save only the new final-stage result.
else:
    print('Reusing:', final_result_path)

# Standalone plot cells reload all four stages from their saved result files.
for plot_cell_index in (14, 16, 17):
    run_notebook_cell(plot_cell_index)


## 10. Additional final plots with Stage 3 at t=741456
Run the next cell alone from the repository root. It reuses stages 512, 16384 and 2097152, fits only the later Stage-3 checkpoint if needed, and exports the three final figures with `_stage3_t741456` in their filenames. Original plots and results are preserved. Probe datasets, settings and theoretical plateaus stay the same.


In [ ]:
# Run this cell alone, including from a fresh kernel at the repository root.
# Fit only t=741456; reuse the other stages and preserve the original figures.
import json
from pathlib import Path

notebook_path = Path.cwd() / 'probing_inside_outside_ternary_mixed_train_test_token_level_depth_12.ipynb'
notebook = json.loads(notebook_path.read_text())
alternative_step = 741456
other_steps = (512, 16384, 2097152)
result_path = Path.cwd() / f'probe_results_step_{alternative_step}_mixed_train_ternary_test_last_token_depth12.pkl'
required_caches = [Path.cwd() / f'probe_results_step_{step}_mixed_train_ternary_test_last_token_depth12.pkl'
                   for step in other_steps]
missing = [str(path) for path in required_caches if not path.is_file()]
if missing:
    raise FileNotFoundError('Run the original analysis first; missing saved stages:\n' + '\n'.join(missing))

run_namespace = {'__name__': '__main__'}
def run_alternative_cell(index, *, plot=False):
    source = ''.join(notebook['cells'][index]['source'])
    # Switch Stage 3 consistently in checkpoint selection, stage labels and theory.
    source = source.replace('262144', str(alternative_step))
    if plot:
        # Keep each original PDF/PNG and export this variant alongside it.
        source = source.replace('.pdf', '_stage3_t741456.pdf').replace('.png', '_stage3_t741456.png')
    exec(compile(source, f'{notebook_path.name}:alternative-cell-{index}', 'exec'), run_namespace)

# Check the cache before deciding whether model/probe work is necessary.
import pickle
cache_valid = False
if result_path.is_file():
    with result_path.open('rb') as handle:
        saved = pickle.load(handle)
    metrics = saved.get('token_probe', saved)
    cache_valid = all(len(metrics.get(metric, [])) == 13
                      for metric in ('test_accuracy', 'test_cross_entropy_normalized'))
if not cache_valid:
    run_alternative_cell(2)  # Identical seeded mixed-train / ternary-test datasets.
    run_alternative_cell(4)  # Includes the compiled-checkpoint loading fix.
    run_namespace['STEPS'] = [alternative_step]
    run_namespace['RECOMPUTE'] = False
    run_alternative_cell(6)  # Fit and save only the alternative Stage 3.
else:
    print('Reusing:', result_path)

# Combined final figure, accuracy panel, and standalone cross-entropy panel.
for plot_cell_index in (14, 16, 17):
    run_alternative_cell(plot_cell_index, plot=True)


## 11. Cross-entropy versus relative layer: D=6 and D=12

Overlay the saved normalized test losses on $k/D$, with embedding at 0 and the final layer at 1. Colors denote Stage 1a (purple), Stage 1b (blue), Stage 2 (orange), and Stage 3 (green); dashed squares denote $D=6$ and solid circles denote $D=12$. The y-axis is logarithmic and the four theoretical normalized loss plateaus are retained. Losses are divided by $\log(8)$, matching the existing figures. The Inside/Outside divider is at $k/D=1/2$ (layers 3 and 6 respectively).

The six-layer sources are `probe_results_step_{step}_mixed_train_ternary_test_last_token.pkl`, identified in `probing_inside_outside_ternary_mixed_train_test_token_level.ipynb`; they contain seven values each. The twelve-layer sources add `_depth12` and contain thirteen values each. The cell checks checkpoint steps, curve lengths, probe settings, sample counts and dataset seeds, and runs independently without retraining.

| Model depth | Stage 1a | Stage 1b | Stage 2 | Stage 3 |
|---|---:|---:|---:|---:|
| D=6 | 512 | 16384 | 262144 | 2965824 |
| D=12 | 512 | 16384 | 262144 | 2097152 |

Stage 3 therefore compares different training steps. A second export uses the notebook's alternative D=12 Stage 2 at t=741456, keeping D=6 Stage 2 at t=262144. These are stage comparisons, not matched-step comparisons at those differing checkpoints. Relative layer aligns layer positions; it does not equalize the models' widths (512 for D=6, 256 for D=12).


In [ ]:
# Reload the current plotting script on every run, independently of kernel state.
from pathlib import Path
import runpy

plot_script = Path.cwd() / 'plot_relative_depth_depth6_vs_depth12.py'
if not plot_script.is_file():
    raise FileNotFoundError(f'Run from the repository root; missing {plot_script}')
print('Plot labels: Stage 1a (purple), Stage 1b (blue), Stage 2 (orange), Stage 3 (green)')
relative_depth_plots = runpy.run_path(str(plot_script))


In [ ]:
# Run independently using the saved probe results; no model loading or refitting.
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

WORKSPACE = Path.cwd()
STEPS = [1024, 262144, 1048576]
MODEL_DEPTH, V = 3, 8
curves_ce = {}
for step in STEPS:
    path = WORKSPACE / f'probe_results_step_{step}_ternary_train_ternary_test_last_token_depth3.pkl'
    if not path.is_file():
        raise FileNotFoundError(f'Run the probe cells first; missing {path}')
    with path.open('rb') as handle:
        saved = pickle.load(handle)
    if (saved.get('experiment') != 'ternary_train_ternary_test_last_token_depth3_v1'
            or saved.get('checkpoint_step') != step):
        raise ValueError(f'Unexpected experiment or checkpoint in {path}')
    curve = np.asarray(saved['token_probe']['test_cross_entropy_normalized'], dtype=float)
    if curve.shape != (MODEL_DEPTH + 1,) or not np.all(np.isfinite(curve)) or np.any(curve <= 0):
        raise ValueError(f'Invalid cross-entropy curve in {path}')
    curves_ce[step] = curve

plt.rcParams.update({'font.size': 9, 'axes.labelsize': 10, 'legend.fontsize': 8,
                     'pdf.fonttype': 42, 'ps.fonttype': 42})
depth = np.arange(MODEL_DEPTH + 1)
fig, ax = plt.subplots(figsize=(3.5, 3.45))
for stage, step in enumerate(STEPS, 1):
    ax.plot(depth, curves_ce[step], marker='o', lw=1.8, ms=3.8,
            label=f'Stage {stage}')
plateaus = {
    r"$\mathcal{L}_{\mathrm{trig}}^{\mathrm{FT}}$": 0.452838,
    r"$\mathcal{L}_2$": 0.205742,
    r"$\mathcal{L}_3$": 0.115976,
}
for label, value in plateaus.items():
    ax.axhline(value, color='black', ls='--', lw=1.05, alpha=.72, zorder=2)
    ax.text(MODEL_DEPTH + .13, value * 1.025, label,
            color='black', fontsize=8.5, va='bottom')
ax.axhline(1, color='grey', ls=':', lw=1, label='Uniform prediction')
ax.set(xticks=depth, xticklabels=[r'emb ($k=0$)', r'$k=1$', r'$k=2$', r'$k=3$'],
       xlim=(-.2, MODEL_DEPTH + .9),
       ylim=(min(min(curve.min() for curve in curves_ce.values()), min(plateaus.values())) * .8,
             max(1, max(curve.max() for curve in curves_ce.values())) * 1.1),
       yscale='log',
       xlabel=r'Transformer layer $k$', ylabel=r'Probe loss $\mathcal{L}_{x_d}^{(k,t)}$',
       title='Fixed-tree training')
ax.grid(alpha=.16, lw=.55)
fig.legend(*ax.get_legend_handles_labels(), loc='upper center', ncol=2,
           frameon=False, bbox_to_anchor=(.5, 1.01), columnspacing=1.0)
fig.tight_layout(rect=(0, 0, 1, .83))
for extension in ('pdf', 'png'):
    path = WORKSPACE / f'token_probe_cross_entropy_ternary_train_ternary_test_across_depth_depth3.{extension}'
    fig.savefig(path, dpi=300)
    print('saved:', path)
plt.show()


## 12. Paper figure: fixed-tree and varying-tree training

Side-by-side panels share the same logarithmic y-axis and normalized loss (cross-entropy divided by log(8)). The left uses the pasted three-layer fixed-tree data; the right uses the six-/twelve-layer varying-tree comparison from Section 11, with Stage 2 at t=262144 for both models. Stage 3 retains the different model-specific steps documented there. Run the cell independently from the repository root. The script reloads on every run; set VARYING_D12_STAGE2 in the script to 741456 for the later checkpoint variant.


In [ ]:
import runpy
# Stage 1a uses t=1024 token-probe caches for both D=6 and D=12.
paper_figure = runpy.run_path('plot_fixed_vs_varying_tree.py')
